In [1]:
import os
import re
import json
import argparse
import pickle

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import average_precision_score, roc_auc_score
import lightgbm as lgb
from sentence_transformers import SentenceTransformer


def parse_args():
    p = argparse.ArgumentParser(description="Train LightGBM on tabular features + LaBSE embeddings")

    p.add_argument("--train-path", type=str,
                    default="public_dataset.csv")
    p.add_argument("--test-path", type=str,
                    default="new_submission_sample (3) (1).csv")
    p.add_argument("--output-dir", type=str, default="model_lgb_labse")

    p.add_argument("--text-col", type=str, default="supplier_room_name")
    p.add_argument("--target-col", type=str, default="target")

    # тот же сплит, что у CatBoost: 15% test, из оставшихся 15% под 8-fold CV
    p.add_argument("--test-size", type=float, default=0.15)
    p.add_argument("--n-folds", type=int, default=8)
    p.add_argument("--seed", type=int, default=42)

    # LaBSE
    p.add_argument("--embedding-model", type=str, default="sentence-transformers/LaBSE")
    p.add_argument("--embedding-batch-size", type=int, default=256)
    p.add_argument("--embedding-device", type=str, default=None,
                    help="'cuda' / 'cpu' / None (автоопределение)")

    # LightGBM
    p.add_argument("--n-estimators", type=int, default=3000)
    p.add_argument("--learning-rate", type=float, default=0.03)
    p.add_argument("--num-leaves", type=int, default=63)
    p.add_argument("--max-depth", type=int, default=-1)
    p.add_argument("--min-child-samples", type=int, default=20)
    p.add_argument("--reg-lambda", type=float, default=5.0)
    p.add_argument("--subsample", type=float, default=0.8)
    p.add_argument("--colsample-bytree", type=float, default=0.8)
    p.add_argument("--early-stopping-rounds", type=int, default=150)

    p.add_argument("--data-fraction", type=float, default=1.0,
                    help="Доля данных для быстрой отладки, 1.0 = весь датасет")

    args, _unknown = p.parse_known_args()
    return args


ARGS = parse_args()
os.makedirs(ARGS.output_dir, exist_ok=True)

TEXT_COL = ARGS.text_col
TARGET_COL = ARGS.target_col
SEED = ARGS.seed

OOF_PROBS_PATH = os.path.join(ARGS.output_dir, "oof_lgb_probs.npy")
OOF_TRUE_PATH = os.path.join(ARGS.output_dir, "oof_lgb_true.npy")
TEST_PROBS_PATH = os.path.join(ARGS.output_dir, "test_lgb_probs.npy")
SUBMISSION_PATH = os.path.join(ARGS.output_dir, "submission_lgb.csv")
FEATURE_COLS_PATH = os.path.join(ARGS.output_dir, "feature_cols_lgb.pkl")
CONFIG_PATH = os.path.join(ARGS.output_dir, "train_config.json")

def preprocess_text(text):
    if not isinstance(text, str) or text.strip() == "":
        return None
    original = text.strip()
    if re.fullmatch(r'[\d\s\W]+', original):
        return None
    text = original.lower()
    text = re.sub(r'[^a-zа-яё0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    if len(text.split()) < 2:
        return None
    return text

EURO_MARKERS = {
    "de": r"\b(doppelzimmer|einzelzimmer|zimmer|familienzimmer|suite|mit)\b",
    "fr": r"\b(chambre|singuliere|vue|avec|lit|familiale|superieure)\b",
    "it": r"\b(camera|matrimoniale|singola|doppia|vista|con|letto|superiore)\b",
    "es": r"\b(habitacion|doble|individual|con|vista|cama|superior)\b",
}


def has_foreign_language(text: str) -> int:
    if not isinstance(text, str) or text.strip() == "":
        return 0
    t = text.lower()
    for pattern in EURO_MARKERS.values():
        if re.search(pattern, t):
            return 1
    return 0


def sample_fraction_df(df: pd.DataFrame, frac: float, seed: int = 42) -> pd.DataFrame:
    if frac >= 1.0:
        return df.reset_index(drop=True)
    n = max(1, int(len(df) * frac))
    return df.sample(n=n, random_state=seed, replace=False).reset_index(drop=True)

def add_features(df_: pd.DataFrame) -> pd.DataFrame:
    df_ = df_.copy()
    text = df_["text_clean"].fillna("")
    orig = df_["supplier_room_name"].fillna("")

    # базовые
    df_["text_length"] = orig.apply(len)
    df_["word_count"] = text.apply(lambda x: len(x.split()))
    df_["unique_ratio"] = text.apply(lambda x: len(set(x.split())) / max(len(x.split()), 1))
    df_["upper_ratio"] = orig.apply(lambda x: sum(1 for c in x if c.isupper()) / max(len(x), 1))
    df_["has_parentheses"] = orig.apply(lambda x: int("(" in x))
    df_["digit_count"] = orig.apply(lambda x: sum(1 for c in x if c.isdigit()))

    # языковые
    df_["ru_ratio"] = text.apply(lambda x: sum(1 for c in x if 'а' <= c <= 'я' or c == 'ё') / max(len(x), 1))
    df_["en_ratio"] = text.apply(lambda x: sum(1 for c in x if 'a' <= c <= 'z') / max(len(x), 1))
    # НОВОЕ: флаг наличия других европейских языков (DE/FR/IT/ES) — проверяем на orig
    df_["has_foreign_lang"] = orig.apply(has_foreign_language)

    bed_configs = [
        "king", "queen", "twin", "double", "single", "bunk",
        "двуспальн", "односпальн", "1 bed", "2 bed",
        "1 кровать", "2 кровати", "2 отдельные",
    ]
    df_["has_bed_config"] = text.apply(lambda x: int(any(b in x for b in bed_configs)))

    room_levels = [
        "standard", "стандарт", "deluxe", "делюкс",
        "superior", "executive", "premium", "премиум",
        "classic", "comfort", "комфорт", "люкс", "economy",
    ]
    df_["has_room_level"] = text.apply(lambda x: int(any(r in x for r in room_levels)))

    view_words = [
        "sea view", "ocean view", "вид на море",
        "city view", "вид на город", "garden view",
        "pool view", "mountain view", " view", " вид",
    ]
    df_["has_view"] = text.apply(lambda x: int(any(v in x for v in view_words)))

    balcony_words = ["balcon", "terrace", "террас", "балкон", "лоджия"]
    df_["has_balcony"] = text.apply(lambda x: int(any(b in x for b in balcony_words)))

    family_words = ["family", "семейн", "child", "детск", "kids"]
    df_["is_family"] = text.apply(lambda x: int(any(f in x for f in family_words)))

    biz = ["has_bed_config", "has_room_level", "has_view", "has_balcony", "is_family"]
    df_["business_completeness"] = df_[biz].sum(axis=1) / len(biz)

    return df_


TABULAR_FEATURE_COLS = [
    "text_length", "word_count", "unique_ratio", "upper_ratio",
    "has_parentheses", "digit_count", "ru_ratio", "en_ratio",
    "has_foreign_lang", 
    "has_bed_config", "has_room_level", "has_view",
    "has_balcony", "is_family", "business_completeness",
    "hotel_positive_rate", "hotel_room_count", "hotel_target_std",
]


def add_hotel_features(df_target: pd.DataFrame, hotel_stats: pd.DataFrame,
                        global_mean: float) -> pd.DataFrame:
    df_target = df_target.merge(hotel_stats, on="hotel_id", how="left")
    df_target["hotel_positive_rate"] = df_target["hotel_positive_rate"].fillna(global_mean)
    df_target["hotel_room_count"] = df_target["hotel_room_count"].fillna(0)
    df_target["hotel_target_std"] = df_target["hotel_target_std"].fillna(0)
    return df_target


def compute_hotel_stats(df_fold_train: pd.DataFrame) -> tuple:
    hotel_stats = df_fold_train.groupby("hotel_id")[TARGET_COL].agg(
        hotel_positive_rate="mean",
        hotel_room_count="count",
        hotel_target_std="std",
    ).reset_index()
    hotel_stats["hotel_target_std"] = hotel_stats["hotel_target_std"].fillna(0)
    global_mean = df_fold_train[TARGET_COL].mean()
    return hotel_stats, global_mean

def build_embeddings(texts, model_name: str, batch_size: int, device):
    print(f"Загружаем эмбеддинг-модель {model_name} (device={device or 'auto'})...")
    model = SentenceTransformer(model_name, device=device)
    embeddings = model.encode(
        list(texts),
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    return embeddings.astype(np.float32)


def main():
    with open(CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump(vars(ARGS), f, ensure_ascii=False, indent=2)

    df = pd.read_csv(ARGS.train_path)
    print("Shape:", df.shape)
    print(df[TARGET_COL].value_counts())

    if ARGS.data_fraction < 1.0:
        df = sample_fraction_df(df, ARGS.data_fraction, SEED)
        print(f"FAST CHECK MODE: data_fraction={ARGS.data_fraction:.2%}, shape={df.shape}")

    df_train_val, df_test = train_test_split(
        df, test_size=ARGS.test_size, random_state=SEED, stratify=df[TARGET_COL]
    )
    df_train_val = df_train_val.reset_index(drop=True)
    df_test = df_test.reset_index(drop=True)
    print(f"train_val: {len(df_train_val)} | test: {len(df_test)}")

    df_train_val["text_clean"] = df_train_val[TEXT_COL].apply(preprocess_text)
    df_test["text_clean"] = df_test[TEXT_COL].apply(preprocess_text)

    before = len(df_train_val)
    df_train_val = df_train_val.dropna(subset=["text_clean"]).reset_index(drop=True)
    df_test["text_clean"] = df_test["text_clean"].fillna("")
    print(f"train_val после очистки: {before} -> {len(df_train_val)}")

    df_train_val = df_train_val.drop_duplicates(
        subset=["hotel_id", TEXT_COL, TARGET_COL]
    ).reset_index(drop=True)
    print(f"train_val после удаления дублей: {len(df_train_val)}")

    train_val_embeddings = build_embeddings(
        df_train_val["text_clean"].tolist(),
        ARGS.embedding_model, ARGS.embedding_batch_size, ARGS.embedding_device,
    )
    test_embeddings = build_embeddings(
        df_test["text_clean"].tolist(),
        ARGS.embedding_model, ARGS.embedding_batch_size, ARGS.embedding_device,
    )
    emb_cols = [f"labse_{i}" for i in range(train_val_embeddings.shape[1])]
    print(f"Эмбеддинги: {train_val_embeddings.shape}")

    y = df_train_val[TARGET_COL].values

    skf = StratifiedKFold(n_splits=ARGS.n_folds, shuffle=True, random_state=SEED)

    oof_probs = np.zeros(len(df_train_val), dtype=np.float32)
    test_probs_folds = np.zeros((ARGS.n_folds, len(df_test)), dtype=np.float32)
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(df_train_val, y)):
        print(f"\n=== Fold {fold + 1}/{ARGS.n_folds} ===")

        fold_train = df_train_val.iloc[tr_idx].copy()
        fold_val = df_train_val.iloc[val_idx].copy()
        fold_test = df_test.copy()

        hotel_stats, global_mean = compute_hotel_stats(fold_train)
        fold_train = add_hotel_features(fold_train, hotel_stats, global_mean)
        fold_val = add_hotel_features(fold_val, hotel_stats, global_mean)
        fold_test = add_hotel_features(fold_test, hotel_stats, global_mean)

        fold_train = add_features(fold_train)
        fold_val = add_features(fold_val)
        fold_test = add_features(fold_test)

        X_tr_tab = fold_train[TABULAR_FEATURE_COLS].fillna(0).values
        X_val_tab = fold_val[TABULAR_FEATURE_COLS].fillna(0).values
        X_test_tab = fold_test[TABULAR_FEATURE_COLS].fillna(0).values

        X_tr = np.hstack([X_tr_tab, train_val_embeddings[tr_idx]])
        X_val = np.hstack([X_val_tab, train_val_embeddings[val_idx]])
        X_test = np.hstack([X_test_tab, test_embeddings])

        y_tr = y[tr_idx]
        y_val = y[val_idx]

        model = lgb.LGBMClassifier(
            n_estimators=ARGS.n_estimators,
            learning_rate=ARGS.learning_rate,
            num_leaves=ARGS.num_leaves,
            max_depth=ARGS.max_depth,
            min_child_samples=ARGS.min_child_samples,
            reg_lambda=ARGS.reg_lambda,
            subsample=ARGS.subsample,
            colsample_bytree=ARGS.colsample_bytree,
            objective="binary",
            random_state=SEED,
            n_jobs=-1,
        )

        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            eval_metric="average_precision",
            callbacks=[
                lgb.early_stopping(ARGS.early_stopping_rounds, verbose=False),
                lgb.log_evaluation(period=200),
            ],
        )

        val_pred = model.predict_proba(X_val)[:, 1]
        oof_probs[val_idx] = val_pred

        fold_pr_auc = average_precision_score(y_val, val_pred)
        fold_roc_auc = roc_auc_score(y_val, val_pred)
        fold_scores.append(fold_pr_auc)
        print(f"Fold {fold + 1}: PR-AUC={fold_pr_auc:.4f} | ROC-AUC={fold_roc_auc:.4f} | "
              f"best_iteration={model.best_iteration_}")

        test_probs_folds[fold] = model.predict_proba(X_test)[:, 1]

        model_path = os.path.join(ARGS.output_dir, f"lgb_fold_{fold}.txt")
        model.booster_.save_model(model_path)

    oof_pr_auc = average_precision_score(y, oof_probs)
    oof_roc_auc = roc_auc_score(y, oof_probs)
    print(f"\n=== OOF (все {ARGS.n_folds} фолдов) ===")
    print(f"PR-AUC:  {oof_pr_auc:.4f}")
    print(f"ROC-AUC: {oof_roc_auc:.4f}")
    print(f"PR-AUC по фолдам: {[round(s, 4) for s in fold_scores]} "
          f"(mean={np.mean(fold_scores):.4f}, std={np.std(fold_scores):.4f})")

    np.save(OOF_PROBS_PATH, oof_probs)
    np.save(OOF_TRUE_PATH, y)

    test_probs_final = test_probs_folds.mean(axis=0)
    np.save(TEST_PROBS_PATH, test_probs_final)

    if "row_id" in df_test.columns:
        row_id = df_test["row_id"]
    elif "Unnamed: 0" in df_test.columns:
        row_id = df_test["Unnamed: 0"]
    else:
        row_id = df_test.index

    submission = pd.DataFrame({"row_id": row_id, "target": test_probs_final})
    submission.to_csv(SUBMISSION_PATH, index=False)

    with open(FEATURE_COLS_PATH, "wb") as f:
        pickle.dump(TABULAR_FEATURE_COLS + emb_cols, f)

    print(f"OOF-вероятности сохранены: {OOF_PROBS_PATH}")
    print(f"Тестовые вероятности сохранены: {TEST_PROBS_PATH}")
    print(f"Submission сохранён: {SUBMISSION_PATH}")


if __name__ == "__main__":
    main()

Shape: (184138, 3)
target
1    95003
0    89135
Name: count, dtype: int64
train_val: 156517 | test: 27621
train_val после очистки: 156517 -> 154820
train_val после удаления дублей: 154820
Загружаем эмбеддинг-модель sentence-transformers/LaBSE (device=auto)...


modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...


=== Fold 8/8 ===
[LightGBM] [Info] Number of positive: 69393, number of negative: 66075
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.262326 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 197787
[LightGBM] [Info] Number of data points in the train set: 135468, number of used features: 786
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.512246 -> initscore=0.048996
[LightGBM] [Info] Start training from score 0.048996
[200]	valid_0's average_precision: 0.922879	valid_0's binary_logloss: 0.378208
[400]	valid_0's average_precision: 0.93626	valid_0's binary_logloss: 0.34266
[600]	valid_0's average_precision: 0.941614	valid_0's binary_logloss: 0.326089
[800]	valid_0's average_precision: 0.944641	valid_0's binary_logloss: 0.31574
[1000]	valid_0's average_precision: 0.946528	valid_0's binary_logloss: 0.308347
[1200]	valid_0's average_precision: 0.948051	valid_0's binary_logloss: 0.302385
[1400]	valid_

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 8: PR-AUC=0.9537 | ROC-AUC=0.9545 | best_iteration=2993


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



=== OOF (все 8 фолдов) ===
PR-AUC:  0.9543
ROC-AUC: 0.9553
PR-AUC по фолдам: [np.float64(0.9531), np.float64(0.9547), np.float64(0.9532), np.float64(0.9568), np.float64(0.9539), np.float64(0.9545), np.float64(0.9546), np.float64(0.9537)] (mean=0.9543, std=0.0011)

✅ OOF-вероятности сохранены: model_lgb_labse/oof_lgb_probs.npy
✅ Тестовые вероятности сохранены: model_lgb_labse/test_lgb_probs.npy
✅ Submission сохранён: model_lgb_labse/submission_lgb.csv


In [4]:
import glob

REAL_SUBMISSION_PATH = ARGS.test_path  
FINAL_SUBMISSION_PATH = os.path.join(ARGS.output_dir, "submission_lgb_FINAL.csv")

real_test = pd.read_csv(REAL_SUBMISSION_PATH)
print("Реальный сэмпл для сабмита:", real_test.shape)

if "row_id" in real_test.columns:
    real_row_id = real_test["row_id"]
elif "Unnamed: 0" in real_test.columns:
    real_row_id = real_test["Unnamed: 0"]
else:
    real_row_id = real_test.index

real_test["text_clean"] = real_test[TEXT_COL].apply(preprocess_text)
real_test["text_clean"] = real_test["text_clean"].fillna("")

df_full = pd.read_csv(ARGS.train_path)
df_train_val_recon, _ = train_test_split(
    df_full, test_size=ARGS.test_size, random_state=SEED, stratify=df_full[TARGET_COL]
)
df_train_val_recon["text_clean"] = df_train_val_recon[TEXT_COL].apply(preprocess_text)
df_train_val_recon = df_train_val_recon.dropna(subset=["text_clean"]).reset_index(drop=True)
df_train_val_recon = df_train_val_recon.drop_duplicates(
    subset=["hotel_id", TEXT_COL, TARGET_COL]
).reset_index(drop=True)

hotel_stats_full, global_mean_full = compute_hotel_stats(df_train_val_recon)

real_test["hotel_id"] = real_test["hotel_id"].astype(str)
hotel_stats_full["hotel_id"] = hotel_stats_full["hotel_id"].astype(str)

real_test = add_hotel_features(real_test, hotel_stats_full, global_mean_full)
real_test = add_features(real_test)

X_real_tab = real_test[TABULAR_FEATURE_COLS].fillna(0).values

real_embeddings = build_embeddings(
    real_test["text_clean"].tolist(),
    ARGS.embedding_model, ARGS.embedding_batch_size, ARGS.embedding_device,
)
X_real = np.hstack([X_real_tab, real_embeddings])

fold_model_paths = sorted(glob.glob(os.path.join(ARGS.output_dir, "lgb_fold_*.txt")))
print(f"Найдено моделей: {len(fold_model_paths)} (ожидалось {ARGS.n_folds})")
assert len(fold_model_paths) == ARGS.n_folds, "Не все фолды найдены — проверь output_dir"

real_preds = np.zeros((len(fold_model_paths), len(real_test)), dtype=np.float32)
for i, path in enumerate(fold_model_paths):
    booster = lgb.Booster(model_file=path)
    real_preds[i] = booster.predict(X_real)

final_probs = real_preds.mean(axis=0)

final_submission = pd.DataFrame({"row_id": real_row_id, "target": final_probs})
final_submission.to_csv(FINAL_SUBMISSION_PATH, index=False)

print(f"\n Финальный submission: {FINAL_SUBMISSION_PATH}")
print("Размер:", final_submission.shape)
print(final_submission.head())

Реальный сэмпл для сабмита: (11000, 3)
Загружаем эмбеддинг-модель sentence-transformers/LaBSE (device=auto)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/LaBSE
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/43 [00:00<?, ?it/s]

Найдено моделей: 8 (ожидалось 8)

✅ Финальный submission: model_lgb_labse/submission_lgb_FINAL.csv
Размер: (11000, 2)
   row_id    target
0       0  0.043992
1       1  0.181038
2       2  0.999914
3       3  0.954955
4       4  0.004805
